In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dill
import pickle
import re

from tqdm.auto import tqdm
from spacy.matcher import PhraseMatcher



c:\Users\ankes\.conda\envs\resume-job-analyzer\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import classification_report

from sklearn.decomposition import TruncatedSVD

from sklearn.linear_model import Lasso
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from xgboost import XGBRanker

from sklearn.model_selection import GroupKFold

In [4]:
from src.features import *
from src.metric import model_evaluation
from src.utils import tech_aliases

In [5]:
with open('../data/skills/processed/all_skills.pkl', 'rb') as f:
    skills = pickle.load(f)

In [6]:
with open('../data/cleaned/train_df.pkl','rb') as f:
    train_df=pickle.load(f)
    
with open('../data/cleaned/val_df.pkl','rb') as f:
    val_df=pickle.load(f)
        
with open('../data/cleaned/test_df.pkl','rb') as f:
    test_df=pickle.load(f)
    


In [ ]:
# train_df['resume_clean']=train_df['resume_text'].apply(lambda x: further_clean_text(x))
# val_df['resume_clean']=val_df['resume_text'].apply(lambda x: further_clean_text(x))
# test_df['resume_clean']=test_df['resume_text'].apply(lambda x: further_clean_text(x))

# train_df['jd_clean']=train_df['job_description_text'].apply(lambda x: further_clean_text(x))
# val_df['jd_clean']=val_df['job_description_text'].apply(lambda x: further_clean_text(x))
# test_df['jd_clean']=test_df['job_description_text'].apply(lambda x: further_clean_text(x))

In [7]:
train_df.head(2)

,index,resume_text,job_description_text,label,resume_len,jd_len,resume_exp,jd_exp,resume_clean,jd_clean
2,2,SummaryI started my construction career in Jun...,Schweitzer Engineering Laboratories (SEL) Infr...,0,872,438,0.0,6.0,summaryi started my construction career in jun...,schweitzer engineering laboratories sel infras...
3,3,SummaryCertified Electrical Foremanwith thirte...,"Mizick Miller & Company, Inc. is looking for a...",0,684,164,8.0,0.0,summarycertified electrical foremanwith thirte...,mizick miller company inc. is looking for a dy...


In [ ]:
# import spacy
# try:
#     spacy.load("en_core_web_md")
# except OSError:
#     print("Downloading spaCy model 'en_core_web_md'...")
#     !python -m spacy download en_core_web_md --quiet
#     import spacy

In [9]:
def build_phrase_matcher(nlp, alias_dict,skills):
    matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
    patterns=[]
    skill_map={}
    # Add alias dict
    for key, values in alias_dict.items():
        patterns.append(nlp.make_doc(key))
        skill_map[key.lower()]=key
        for v in values:
            patterns.append(nlp.make_doc(v))
            skill_map[v.lower()]=key

     # 2. Add remaining skills (from Kaggle)
    for skill in skills:
        if skill.lower() not in skill_map:
            patterns.append(nlp.make_doc(skill))
            skill_map[skill.lower()] = skill


    matcher.add("SKILLS", patterns)
    return matcher,skill_map


def single_pass_pipeline(texts, matcher,skill_map, nlp):
    processed_texts = []
    print("Applying alias normalization and phrase matching...")


    print("Processing documents with spaCy...")
    for doc in tqdm(nlp.pipe(texts, batch_size=200,n_process=2), total=len(texts)):
        raw_matches = matcher(doc)
        spans = [(match_id, start, end) for match_id, start, end in raw_matches]
        filtered_spans = filter_spans([doc[start:end] for _, start, end in spans])

        span_dict = {}
        match_map = {(start, end): match_id for match_id, start, end in spans}

        for span in filtered_spans:
            match_id = match_map[(span.start, span.end)]
            normalize_skill=skill_map.get(span.text.lower(),span.text.lower())
            span_dict[span.start] = (span.end, normalize_skill)

        tokens = []
        i = 0

        while i < len(doc):
            if i in span_dict:
                end_idx, label = span_dict[i]
                tokens.append(label.replace(" ","_"))
                i = end_idx
                continue

            token = doc[i]

            if (token.is_stop or token.is_punct or token.is_space or
                token.like_url or token.like_email):
                i += 1
                continue

            if token.pos_ not in ['NOUN', 'VERB', 'ADJ', 'PROPN']:
                i += 1
                continue

            lemma = token.lemma_.lower()

            if len(lemma) > 2 :
                tokens.append(lemma)

            i += 1

        processed_texts.append(" ".join(tokens))

    return processed_texts


def preprocess_pipeline(texts, alias_dict,skills):
    nlp = spacy.load("en_core_web_md",disable=["parser", "ner"])
    print("Building phrase matcher...")
    matcher,skill_map =build_phrase_matcher(nlp, alias_dict,skills)
    return single_pass_pipeline(texts,matcher,skill_map,nlp)


In [ ]:
# resume_processed_train = preprocess_pipeline(train_df['resume_text'].tolist(), tech_aliases,skills)
# job_description_processed_train = preprocess_pipeline(train_df['job_description_text'].tolist(), tech_aliases,skills)

# job_description_processed_val = preprocess_pipeline(val_df['job_description_text'].tolist(), tech_aliases,skills)
# resume_processed_val = preprocess_pipeline(val_df['resume_text'].tolist(), tech_aliases,skills)


# resume_processed_test = preprocess_pipeline(test_df['resume_text'].tolist(), tech_aliases,skills)
# job_description_processed_test = preprocess_pipeline(test_df['job_description_text'].tolist(), tech_aliases,skills)


In [10]:
preprocessed_directory_path='../data/preprocessed'

In [11]:
resume_processed_train = pickle.load(open(preprocessed_directory_path + '/train_resume.pkl','rb'))
jd_processed_train = pickle.load(open(preprocessed_directory_path + '/train_jd.pkl','rb'))

jd_processed_val = pickle.load(open(preprocessed_directory_path + '/val_jd.pkl','rb'))
resume_processed_val = pickle.load(open(preprocessed_directory_path + '/val_resume.pkl','rb'))

resume_processed_test = pickle.load(open(preprocessed_directory_path + '/test_resume.pkl','rb'))
jd_processed_test = pickle.load(open(preprocessed_directory_path + '/test_jd.pkl','rb'))


In [ ]:
# with open('../data/processed/preprocessed_train_df.pkl','wb') as f:
#     pickle.dump(train_df,f)

# with open('../data/processed/preprocessed_val_df.pkl','wb') as f:
#     pickle.dump(val_df,f)

# with open('../data/processed/preprocessed_test_df.pkl','wb') as f:
#     pickle.dump(test_df,f)

In [12]:
train_df['resume_clean']=resume_processed_train
train_df['jd_clean']=jd_processed_train

val_df['resume_clean']=resume_processed_val
val_df['jd_clean']=jd_processed_val

test_df['resume_clean']=resume_processed_test
test_df['jd_clean']=jd_processed_test

In [13]:
train_df.head()

,index,resume_text,job_description_text,label,resume_len,jd_len,resume_exp,jd_exp,resume_clean,jd_clean
2,2,SummaryI started my construction career in Jun...,Schweitzer Engineering Laboratories (SEL) Infr...,0,872,438,0.0,6.0,summaryi start construction career june jackso...,schweitzer engineering laboratory sel infrastr...
3,3,SummaryCertified Electrical Foremanwith thirte...,"Mizick Miller & Company, Inc. is looking for a...",0,684,164,8.0,0.0,summarycertifie electrical foremanwith year ex...,mizick miller company inc look dynamic individ...
5,5,"SummarySolution-oriented, results-driven strat...",\n\nResponsibilitiesLead and provide day-to-da...,0,585,306,3.0,6.0,summarysolution orient result drive strategic ...,responsibilitieslead provide day day support a...
6,6,SummaryA position in a company that will utili...,"Senior Salesforce Software Engineer, Salesforc...",0,463,101,0.0,0.0,summarya position company utilize ability acco...,senior salesforce software engineer salesforce...
7,7,SummaryTo participate as a team member in a dy...,***W2 ONLY** \nSoftware Engineer (12-month con...,0,295,169,0.0,1.0,participate as team member dynamic work enviro...,software engineer month contract remote team b...


In [14]:
train_df=train_df.copy()

In [15]:
train_df = train_df.sort_values('job_description_text').reset_index(drop=True)
val_df = val_df.sort_values('job_description_text').reset_index(drop=True)

In [16]:
y_train=train_df['label']
y_val=val_df['label']

In [17]:
vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,2),
    stop_words='english'
)

vectorizer.fit(pd.concat([train_df['resume_clean'],train_df['jd_clean']]))
X_train_resume = vectorizer.transform(train_df['resume_clean'])
X_train_jd = vectorizer.transform(train_df['jd_clean'])

X_val_resume = vectorizer.transform(val_df['resume_clean'])
X_val_jd = vectorizer.transform(val_df['jd_clean'])


In [18]:
train_sim = cosine_similarity(X_train_resume, X_train_jd).diagonal().reshape(-1,1)
val_sim = cosine_similarity(X_val_resume, X_val_jd).diagonal().reshape(-1,1)


In [19]:
skill_overlap_train=skill_overlap(train_df['resume_clean'],train_df['jd_clean'],skills)
skill_overlap_val=skill_overlap(val_df['resume_clean'],val_df['jd_clean'],skills)


In [20]:
exp_gap_train=train_df['resume_exp']-train_df['jd_exp']
exp_gap_val=val_df['resume_exp']-val_df['jd_exp']

exp_match_train=train_df['resume_exp']/(train_df['jd_exp']+1)
exp_match_val=val_df['resume_exp']/(val_df['jd_exp']+1)

exp_enough_train=train_df['resume_exp']>=train_df['jd_exp'].astype(int)
exp_enough_val=val_df['resume_exp']>=val_df['jd_exp'].astype(int)


In [21]:
X_train_features = np.hstack([
    train_sim,
    skill_overlap_train,
    exp_gap_train.to_numpy().reshape(-1,1),
])

X_val_features = np.hstack([
    val_sim,
    skill_overlap_val,
    exp_gap_val.to_numpy().reshape(-1,1),
])

In [22]:
pd.DataFrame(X_train_features).corr()

,0,1,2,3
0,1.000000,0.469717,0.433625,-0.113564
1,0.469717,1.000000,0.382939,-0.131326
2,0.433625,0.382939,1.000000,-0.020239
3,-0.113564,-0.131326,-0.020239,1.000000


In [23]:
X_train_interaction = X_train_resume.multiply(X_train_jd)
X_val_interaction = X_val_resume.multiply(X_val_jd)


In [24]:
svd = TruncatedSVD(n_components=30, random_state=42)
X_train_svd = svd.fit_transform(X_train_interaction)
X_val_svd = svd.transform(X_val_interaction)

X_train_combine = np.hstack([X_train_svd, X_train_features])
X_val_combine = np.hstack([X_val_svd, X_val_features])

scaler = StandardScaler()
X_train_final = scaler.fit_transform(X_train_combine)
X_val_final = scaler.transform(X_val_combine)


In [25]:
models={"SVM":SVC(class_weight="balanced",probability=True),
        "LogisticRegression":LogisticRegression(class_weight="balanced",max_iter=1000),
        "RandomForest": RandomForestClassifier(n_estimators=200,class_weight="balanced", random_state=42),
        "XGB":XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.1)}

In [26]:
def train_eval_classify(X_train, X_test, y_train, y_test, model, df):
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)

  print("Unique predictions:", set(y_pred))
  print("Variance:", np.var(y_pred))

  y_prob = model.predict_proba(X_test)
  scores=y_prob[:,2]

  print(classification_report(y_test, y_pred))

  print("Metrics:")
  model_evaluation(scores,df,'jd_clean')

In [27]:
for name,model_obj in models.items():
    print(f"\n--- Evaluating Model: {name} ","-"*50)
    train_eval_classify(X_train_final, X_val_final, y_train, y_val, model_obj, val_df)


--- Evaluating Model: SVM  --------------------------------------------------
Unique predictions: {np.int64(0), np.int64(1), np.int64(2)}
Variance: 0.6670675803975213
              precision    recall  f1-score   support

           0       0.64      0.75      0.69       613
           1       0.37      0.27      0.31       285
           2       0.25      0.23      0.24       269

    accuracy                           0.51      1167
   macro avg       0.42      0.42      0.42      1167
weighted avg       0.49      0.51      0.50      1167

Metrics:
Spearman: 0.34833618609989997
Top-3 Accuracy: 0.9047619047619048
NDCG: 0.6784262187890833
MRR: 0.8092261904761905
MAP: 0.7552649656377377

--- Evaluating Model: LogisticRegression  --------------------------------------------------
Unique predictions: {np.int64(0), np.int64(1), np.int64(2)}
Variance: 0.6091127837878122
              precision    recall  f1-score   support

           0       0.64      0.69      0.67       613
           1

In [34]:
xgb_rank=XGBRanker(objective='rank:ndcg',learning_rate=0.05,max_depth=7, n_estimators=300,
                  subsample=0.8,colsample_bytree=0.8)

group_train=train_df.groupby('jd_clean').size().to_list()
xgb_rank.fit(X_train_final,y_train,group=group_train)
scores=xgb_rank.predict(X_val_final)

In [ ]:
print("Evaluating on Full Validation Df")
metrics=model_evaluation(scores,val_df,'jd_clean')
print(metrics)

Evaluating on Full Validation Df
Spearman: 0.37050270328839935
Top-3 Accuracy: 1.0
NDCG: 0.678915794382244
MRR: 0.8154761904761904
MAP: 0.7674271585798028


In [36]:
imp_feature=xgb_rank.feature_importances_
cosine_imp=imp_feature[0]
skill_overlap_imp=imp_feature[1:4]
exp_gap_imp=imp_feature[4]
svd_imp=imp_feature[5:].sum()

In [37]:
print("Cosine:", cosine_imp)
print("Skill_overlap:", skill_overlap_imp.sum())
print("Exp_gap:",exp_gap_imp)
print("SVD (semantic):", svd_imp)

Cosine: 0.03364804
Skill_overlap: 0.1052749
Exp_gap: 0.028403798
SVD (semantic): 0.83267325


In [ ]:
kf=GroupKFold(n_splits=5)
ndcg_scores,spearman_scores,topk_scores,mrr_scores,map_scores=[],[],[],[],[]

for fold, (train_split, val_split) in enumerate(
    kf.split(X_train_final, y_train, groups=train_df['jd_clean'])):


    model=XGBRanker(objective='rank:ndcg',learning_rate=0.05,max_depth=7,
                       n_estimators=300,subsample=0.8,colsample_bytree=0.8)

    group_train=train_df.iloc[train_split].groupby('jd_clean').size().to_list()
    
    model.fit(X_train_final[train_split],y_train.iloc[train_split],group=group_train)
    xgb_scores=model.predict(X_train_final[val_split])

    print(f"fr{fold+1}:\n")
    metrics=model_evaluation(xgb_scores,train_df.iloc[val_split],"jd_clean")
    
    print("NDCG:",metrics['ndcg_val'])
    print("MAP:",metrics['map_score'])
    print("MRR:",metrics['mrr_score'])

    spearman_scores.append(metrics['spearman_score'])
    topk_scores.append(metrics['topk_score'])
    ndcg_scores.append(metrics['ndcg_val'])
    map_scores.append(metrics['map_score'])
    mrr_scores.append(metrics['mrr_score'])
    print("-"*100)

print(f"\nCV NDC:{np.mean(ndcg_scores):.4f}±{np.std(ndcg_scores):.4f}")
print(f"CV Spearma:{np.mean(spearman_scores):.4f}±{np.std(spearman_scores):.4f}")
print(f"CV Top-3 Accurac:{np.mean(topk_scores):.4f}±{np.std(topk_scores):.4f}")
print(f"CV MA:{np.mean(map_scores):.4f}±{np.std(map_scores):.4f}")
print(f"CV MR:{np.mean(mrr_scores):.4f}±{np.std(mrr_scores):.4f}")



fr1:

Spearman: 0.2967474099928556
Top-3 Accuracy: 0.9565217391304348
NDCG: 0.6329376524547347
MRR: 0.8445945945945945
MAP: 0.7587048360947609
----------------------------------------------------------------------------------------------------
fr2:

Spearman: 0.373850184653062
Top-3 Accuracy: 0.9090909090909091
NDCG: 0.6972705534655239
MRR: 0.875
MAP: 0.7614052526442723
----------------------------------------------------------------------------------------------------
fr3:

Spearman: 0.25141497064310825
Top-3 Accuracy: 0.7894736842105263
NDCG: 0.6536374824603335
MRR: 0.8262195121951219
MAP: 0.7180629562996201
----------------------------------------------------------------------------------------------------
fr4:

Spearman: 0.31667775413537097
Top-3 Accuracy: 0.8666666666666667
NDCG: 0.6633913321462226
MRR: 0.7736353077816492
MAP: 0.7174454549147706
----------------------------------------------------------------------------------------------------
fr5:

Spearman: 0.2922381340655541
T

In [39]:
eval_df=val_df.copy()
eval_df['score']=scores

In [ ]:
false_neg = eval_df[(eval_df['label'] == 2) & (eval_df['score'] < -0.3)]

print(f"Good Fit resumes scoring below -0.3: {len(false_neg)}")
print("\nSample false_neg resumes:")

i=0
for jd,group in false_neg.groupby('jd_clean'):
    print("-"*100)
    print(f"JD: {jd[:200]}")
    for _,row in group.head(3).iterrows():
        print(f"Index:{row['index']}")
        print(f"Score: {row['score']:.3f}")
        print(f"Resume: {row['resume_text'][:300]}\n")
    i+=1
    if i==3:
        break
        

Good Fit resumes scoring below -0.3: 159

Sample false_neg resumes:
----------------------------------------------------------------------------------------------------
JD: alphabet moonshot factory diverse group inventor entrepreneur build launch technologies aim improve life million billion people goal impact world intractable problem improvement approach project aspir
Index:5080
Score: -0.563
Resume: Career OverviewMerrimack College Alumnus possessing a strong aptitude and interest in topics of Mathematics and Finance. Currently employed at AIR Worldwide as an SQA Engineer III. Additionally I have further developed my skill sets and broadened my professional expertise through two internships wit

Index:4732
Score: -0.719
Resume: Summary•        
Over
Three years of extensive experience as a Front-End UI Developer with solid
understanding of database designing, development and installation of different
modules. 

•        
Professional
understanding of System development life cycle (

In [41]:
false_pos = eval_df[
    (eval_df['label'] == 0) & (eval_df['score'] > 1)]

print(f"Bad Fit resumes scoring above 1: {len(false_pos)}")
print("\nSample false_pos resumes:")

i=0
for jd,group in false_pos.groupby('jd_clean'):
    print("-"*100)
    print(f"JD: {jd[:300]}")
    for _,row in group.head(2).iterrows():
        print(f"Index:{row['index']}")
        print(f"Score: {row['score']:.3f}")
        print(f"Resume: {row['resume_text'][:500]}\n")
    i+=1
    if i==3:
        break
        

Bad Fit resumes scoring above 1: 13

Sample false_pos resumes:
----------------------------------------------------------------------------------------------------
JD: abhishek singh recruiter talent network sole agency recruitment sourcing century technologies inc tscti leverage extensive network resources century technologies inc tscti midsize temporary staffing company fte presence state serve direct federal state local commercial clients tscti customers includ
Index:2804
Score: 1.079
Resume: SummaryAccomplished and results-orientedfinance professionalwho consistently meets deadlines and increases department revenue. Highly skilled at increasing productivity through detailed cost analysis.
HighlightsMicrosoft Office :	Intermediate in all the Microsoft Office components( Excel, Word, PowerPoint, Outlook and Access); Very familiar with ( Macros, V-look ups, calculating formulas and manipulating reports as well as smart view) Running query reports and creating reports.SAP PeopleSoft: N

In [ ]:
print("\nScore spread within groups:")
spreads = []
for jd, group in eval_df.groupby('jd_clean'):
    if len(group) > 1:
        spreads.append(group['score'].max() - group['score'].min())

print(f"Mean spread: {np.mean(spreads)}")
print(f"% groups with spread < 0.1:"f"{(np.array(spreads) < 0.1).mean()}")


Score spread within groups:
Mean spread: 2.485
% groups with spread < 0.1:0.000


In [ ]:
X_test_resume = vectorizer.transform(test_df['resume_clean'])
X_test_jd = vectorizer.transform(test_df['jd_clean'])

test_sim = cosine_similarity(X_test_resume, X_test_jd).diagonal().reshape(-1,1)

skill_overlap_test=skill_overlap(test_df['resume_clean'],test_df['jd_clean'],skills)

exp_gap_test=test_df['resume_exp']-test_df['jd_exp']

X_test_features = np.hstack([
    test_sim,
    skill_overlap_test,
    exp_gap_test.to_numpy().reshape(-1,1),
])

X_test_interaction = X_test_resume.multiply(X_test_jd)

X_test_svd = svd.transform(X_test_interaction)

X_test_combine = np.hstack([X_test_svd, X_test_features])

X_test_final = scaler.transform(X_test_combine)

final_scores=xgb_rank.predict(X_test_final)

metrics=model_evaluation(final_scores,test_df,'jd_clean')
print(metrics)

Spearman: 0.3048802601616147
Top-3 Accuracy: 0.8571428571428571
NDCG: 0.6370559795760189
MRR: 0.8366120218579236
MAP: 0.7460503612974287


In [ ]:
#Finding->
# Data have noise , as tfidf try to lean common word so many resume jd pair have many similar word
# but not thieir semantic mean fully whole diffrent 
# also many jd resume may have noise as jd req electrical engineerinal and in resume chemical engirence
# and its related around work but still selected
# also many pair in boderline it may selected and may not selected it depend entirely on hr
#modle able to somewhat diffrentiate bw fit and no fi as within spread is high
